# RetainAI — Block 4: Tool-Using LLM Agent on Amazon Bedrock

The agent ties every earlier block together. For a customer (and optionally their message) it:

1. Pulls the profile and a **live churn score from the production SageMaker endpoint** (Block 2), with SHAP reasons (Block 1)
2. Classifies the customer's message with the **intent model** (Block 3)
3. Gets eligible offers from a **policy engine enforced in code**, and can search the **retention policy (RAG)**
4. Chooses an offer and drafts a personalized message

Then we **evaluate** it like a production system: programmatic guardrail checks, LLM-as-judge quality scores,
latency, and token usage, comparing two models.

**Design principle:** the LLM decides *what to say*; code decides *what is allowed*.

In [1]:
%pip install -q shap xgboost scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1. Setup

In [2]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, "src")
from agent import Toolbox, Agent, validate, judge, AGENT_MODEL, JUDGE_MODEL, CHALLENGER_MODEL

tools = Toolbox()
active = tools.customers[tools.customers["Churn"] == 0]
print(f"Loaded {len(tools.customers):,} customers | endpoint: {tools.endpoint} | policy sections: {len(tools.policy_chunks)}")
print("Agent:", AGENT_MODEL, "| Judge:", JUDGE_MODEL, "| Challenger:", CHALLENGER_MODEL)

Loaded 7,043 customers | endpoint: retainai-churn-serverless | policy sections: 6
Agent: us.amazon.nova-2-lite-v1:0 | Judge: us.meta.llama3-3-70b-instruct-v1:0 | Challenger: us.amazon.nova-micro-v1:0


## 2. Test each tool directly (before any LLM is involved)

In [3]:
cid = active.sort_values("revenue_at_risk_12m", ascending=False).index[0]
show = lambda name, out: print(f"--- {name}\n{json.dumps(out, indent=2, default=str)}\n")

show("get_customer_profile", tools.get_customer_profile(cid))
show("score_churn_risk (live endpoint)", tools.score_churn_risk(cid))
show("classify_message", tools.classify_message("I want to cancel, the internet keeps dropping"))
show("recommend_offer", tools.recommend_offer(cid))
show("search_policy", tools.search_policy("maximum discount allowed"))

--- get_customer_profile
{
  "customer_id": "2081-VEYEH",
  "tenure_months": 3,
  "contract": "Month-to-month",
  "monthly_charges": 107.95,
  "internet_service": "Fiber optic",
  "payment_method": "Electronic check",
  "active_add_ons": [
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
  ],
  "segment": 4,
  "billing_anomaly_flag": false
}

--- score_churn_risk (live endpoint)
{
  "churn_probability": 0.6764,
  "risk_band": "high",
  "revenue_at_risk_12m": 876.19,
  "top_risk_drivers": [
    "is_month_to_month = 1",
    "tenure = 3",
    "InternetService = Fiber optic"
  ],
  "scored_by": "retainai-churn-serverless"
}

--- classify_message
{
  "intent": "delete_account",
  "confidence": 0.488,
  "is_churn_signal": true
}

--- recommend_offer
{
  "eligible_offers": [
    {
      "code": "NO_OFFER",
      "description": "Friendly check-in, no incentive"
    },
    {
      "code": "ADDON_TRIAL",
      "add_on": "OnlineSecurity",
   

## 3. Run the agent once and inspect its reasoning trace

In [4]:
agent = Agent(tools, model_id=AGENT_MODEL)
result = agent.run(cid, "Honestly my bill is way too high for what I get. Thinking about switching providers.")

for step in result["trace"]:
    print(f"🔧 {step['tool']}({json.dumps(step['input'])})")
print(f"\nTurns: {result['turns']} | Latency: {result['latency_s']}s | Tokens: {result['usage']}\n")
print(json.dumps(result["output"], indent=2))
print("\nGuardrail checks:", validate(result, tools))

🔧 get_customer_profile({"customer_id": "2081-VEYEH"})
🔧 score_churn_risk({"customer_id": "2081-VEYEH"})
🔧 classify_message({"text": "Honestly my bill is way too high for what I get. Thinking about switching providers."})
🔧 recommend_offer({"customer_id": "2081-VEYEH"})
🔧 search_policy({"query": "high risk month-to-month customer with 3 months tenure and fiber service"})

Turns: 4 | Latency: 3.02s | Tokens: {'inputTokens': 7644, 'outputTokens': 296}

{
  "risk_level": "high",
  "key_reasons": [
    "month-to-month",
    "tenure 3 months",
    "Fiber optic service"
  ],
  "offer_code": "LOYALTY_DISCOUNT",
  "escalate_to_human": false,
  "customer_message": "I hear you about the bill feeling high. To help, I can offer you 15% off your monthly charge for the next 6 months. This should bring more value to your Fiber service while you keep all your current add-ons. Let me know if you'd like to get this discount applied."
}

Guardrail checks: {'valid_json': True, 'called_required_tools': True

## 4. Evaluation scenarios

A mix of risk levels and situations, including cases where the right answer is *not* to offer a discount.

In [5]:
by_risk = active.sort_values("churn_prob", ascending=False)
high = by_risk[by_risk["churn_prob"] >= 0.6].index
mid = by_risk[(by_risk["churn_prob"] >= 0.3) & (by_risk["churn_prob"] < 0.6)].index
low = by_risk[by_risk["churn_prob"] < 0.15].index
anomaly = active[(active["billing_anomaly"] == 1) & (active["churn_prob"] >= 0.3)].index
no_internet = active[(active["InternetService"] == "No") & (active["churn_prob"] >= 0.3)].index

def first(idx, n=0):
    return idx[n] if len(idx) > n else by_risk.index[n]

scenarios = [
    {"name": "High risk, proactive outreach", "customer_id": first(high, 1), "message": None},
    {"name": "High risk, price complaint", "customer_id": first(high, 2),
     "message": "My bill keeps going up and the service hasn't changed. Why am I paying this much?"},
    {"name": "High risk, wants to close account", "customer_id": first(high, 3),
     "message": "Third outage this month. I want to close my account."},
    {"name": "High risk, double charge", "customer_id": first(high, 4),
     "message": "You charged me twice this month and nobody has helped me."},
    {"name": "Medium risk, proactive", "customer_id": first(mid, 0), "message": None},
    {"name": "Medium risk, simple question", "customer_id": first(mid, 1),
     "message": "Where can I find my latest invoice?"},
    {"name": "Low risk, proactive (should be NO_OFFER)", "customer_id": first(low, 0), "message": None},
    {"name": "Low risk, simple question", "customer_id": first(low, 1),
     "message": "How do I update my payment method?"},
    {"name": "Billing anomaly flagged", "customer_id": first(anomaly, 0),
     "message": "My bill looks wrong this month."},
    {"name": "Phone-only customer at risk", "customer_id": first(no_internet, 0), "message": None},
]
pd.DataFrame(scenarios)

,name,customer_id,message
0,"High risk, proactive outreach",1941-HOSAM,None
1,"High risk, price complaint",0021-IKXGC,My bill keeps going up and the service hasn't ...
2,"High risk, wants to close account",4912-PIGUY,Third outage this month. I want to close my ac...
3,"High risk, double charge",7577-SWIFR,You charged me twice this month and nobody has...
4,"Medium risk, proactive",0480-BIXDE,None
5,"Medium risk, simple question",7729-JTEEC,Where can I find my latest invoice?
6,"Low risk, proactive (should be NO_OFFER)",4797-MIWUM,None
7,"Low risk, simple question",9060-HJJRW,How do I update my payment method?
8,Billing anomaly flagged,0771-WLCLA,My bill looks wrong this month.
9,Phone-only customer at risk,7310-EGVHZ,None


## 5. Evaluate the agent model

In [6]:
def evaluate(model_id, scenarios):
    runner, rows, runs = Agent(tools, model_id=model_id), [], []
    for s in scenarios:
        res = runner.run(s["customer_id"], s["message"])
        checks = validate(res, tools)
        scores = judge(res, tools) if res["output"] else {}
        out = res["output"] or {}
        rows.append({"scenario": s["name"], "offer": out.get("offer_code"), "escalate": out.get("escalate_to_human"),
                     **{k: v for k, v in checks.items()}, "judge_avg": scores.get("judge_avg"),
                     "judge_comment": scores.get("comment"), "latency_s": res["latency_s"],
                     "input_tokens": res["usage"]["inputTokens"], "output_tokens": res["usage"]["outputTokens"],
                     "tool_calls": len(res["trace"])})
        runs.append(res)
        print(f"{'✅' if checks['compliant'] else '❌'} {s['name']:<42} offer={out.get('offer_code')}  judge={scores.get('judge_avg')}")
    return pd.DataFrame(rows), runs

agent_df, agent_runs = evaluate(AGENT_MODEL, scenarios)
agent_df.drop(columns=["judge_comment"])

✅ High risk, proactive outreach              offer=CONTRACT_UPGRADE  judge=3.75
✅ High risk, price complaint                 offer=LOYALTY_DISCOUNT  judge=3.75
✅ High risk, wants to close account          offer=NO_OFFER  judge=3.5
❌ High risk, double charge                   offer=BILLING_REVIEW  judge=3.75
✅ Medium risk, proactive                     offer=CONTRACT_UPGRADE  judge=3.75
✅ Medium risk, simple question               offer=NO_OFFER  judge=3.75
✅ Low risk, proactive (should be NO_OFFER)   offer=NO_OFFER  judge=3.0
✅ Low risk, simple question                  offer=NO_OFFER  judge=3.25
✅ Billing anomaly flagged                    offer=BILLING_REVIEW  judge=3.75
✅ Phone-only customer at risk                offer=CONTRACT_UPGRADE  judge=4.0


,scenario,offer,escalate,valid_json,called_required_tools,offer_is_eligible,discount_within_policy,no_internal_terms,escalation_correct,under_120_words,compliant,judge_avg,latency_s,input_tokens,output_tokens,tool_calls
0,"High risk, proactive outreach",CONTRACT_UPGRADE,False,True,True,True,True,True,True,True,True,3.75,1.70,3338,223,3
1,"High risk, price complaint",LOYALTY_DISCOUNT,False,True,True,True,True,True,True,True,True,3.75,1.72,3433,253,4
2,"High risk, wants to close account",NO_OFFER,True,True,True,True,True,True,True,True,True,3.50,2.55,7430,260,5
3,"High risk, double charge",BILLING_REVIEW,True,True,True,False,True,True,True,True,False,3.75,2.94,7562,274,5
4,"Medium risk, proactive",CONTRACT_UPGRADE,False,True,True,True,True,True,True,True,True,3.75,1.72,3342,217,3
5,"Medium risk, simple question",NO_OFFER,False,True,True,True,True,True,True,True,True,3.75,2.26,5145,204,4
6,"Low risk, proactive (should be NO_OFFER)",NO_OFFER,False,True,True,True,True,True,True,True,True,3.00,1.40,3180,179,3
7,"Low risk, simple question",NO_OFFER,False,True,True,True,True,True,True,True,True,3.25,1.69,3261,209,4
8,Billing anomaly flagged,BILLING_REVIEW,False,True,True,True,True,True,True,True,True,3.75,1.77,3461,207,4
9,Phone-only customer at risk,CONTRACT_UPGRADE,False,True,True,True,True,True,True,True,True,4.00,1.48,3284,204,3


In [7]:
# Read a few of the actual messages the agent wrote
for res in agent_runs[:4]:
    out = res["output"] or {}
    print(f"[{res['customer_id']}] {res['message'] or '(proactive)'}")
    print(f"  offer: {out.get('offer_code')} | escalate: {out.get('escalate_to_human')}")
    print(f"  -> {out.get('customer_message')}\n")

[1941-HOSAM] (proactive)
  offer: CONTRACT_UPGRADE | escalate: False
  -> Hi there! I see you’re new to us and enjoying Fiber optic internet. To make things even better, we’d like to offer you a 1-year contract with 10% off your monthly charge. This will lock in your rate and give you added peace of mind. Would you like to hear more?

[0021-IKXGC] My bill keeps going up and the service hasn't changed. Why am I paying this much?
  offer: LOYALTY_DISCOUNT | escalate: False
  -> I understand your concern about the rising bill. To help, I’d like to offer you 15% off your monthly charge for the next 6 months. This should ease the cost while keeping your service unchanged. Let me know if you’d like to proceed.

[4912-PIGUY] Third outage this month. I want to close my account.
  offer: NO_OFFER | escalate: True
  -> I’m very sorry to hear about the outages. A retention specialist will contact you within one business day to discuss your concerns and explore solutions.

[7577-SWIFR] You charged

## 6. Challenger: is a cheaper model good enough?

In [8]:
challenger_df, challenger_runs = evaluate(CHALLENGER_MODEL, scenarios)

def summarize(df, model):
    return {"model": model.split(".")[-1], "compliance_rate": df["compliant"].mean(),
            "avg_judge_score": df["judge_avg"].mean(), "avg_latency_s": df["latency_s"].mean(),
            "avg_input_tokens": df["input_tokens"].mean(), "avg_output_tokens": df["output_tokens"].mean(),
            "avg_tool_calls": df["tool_calls"].mean()}

comparison = pd.DataFrame([summarize(agent_df, AGENT_MODEL), summarize(challenger_df, CHALLENGER_MODEL)])
comparison.round(3)

✅ High risk, proactive outreach              offer=CONTRACT_UPGRADE  judge=3.0
✅ High risk, price complaint                 offer=ADDON_TRIAL  judge=3.5
✅ High risk, wants to close account          offer=NO_OFFER  judge=3.25
✅ High risk, double charge                   offer=ADDON_TRIAL  judge=3.25
✅ Medium risk, proactive                     offer=ADDON_TRIAL  judge=3.25
✅ Medium risk, simple question               offer=NO_OFFER  judge=3.5
✅ Low risk, proactive (should be NO_OFFER)   offer=NO_OFFER  judge=4.0
✅ Low risk, simple question                  offer=NO_OFFER  judge=3.25
✅ Billing anomaly flagged                    offer=BILLING_REVIEW  judge=4.0
✅ Phone-only customer at risk                offer=CONTRACT_UPGRADE  judge=3.25


,model,compliance_rate,avg_judge_score,avg_latency_s,avg_input_tokens,avg_output_tokens,avg_tool_calls
0,nova-2-lite-v1:0,0.9,3.625,1.923,4343.6,223.0,3.8
1,nova-micro-v1:0,1.0,3.425,3.594,5124.2,682.0,4.4


In [9]:
# Which guardrails failed, per model?
check_cols = ["valid_json", "called_required_tools", "offer_is_eligible", "discount_within_policy",
              "no_internal_terms", "escalation_correct", "under_120_words"]
pd.DataFrame({AGENT_MODEL.split(".")[-1]: agent_df[check_cols].mean(),
              CHALLENGER_MODEL.split(".")[-1]: challenger_df[check_cols].mean()}).round(2)

,nova-2-lite-v1:0,nova-micro-v1:0
valid_json,1.0,1.0
called_required_tools,1.0,1.0
offer_is_eligible,0.9,1.0
discount_within_policy,1.0,1.0
no_internal_terms,1.0,1.0
escalation_correct,1.0,1.0
under_120_words,1.0,1.0


## 7. Save results and example transcripts

In [10]:
os.makedirs("artifacts", exist_ok=True); os.makedirs("docs", exist_ok=True)
agent_df.assign(model=AGENT_MODEL).to_csv("artifacts/agent_eval.csv", index=False)
challenger_df.assign(model=CHALLENGER_MODEL).to_csv("artifacts/agent_eval_challenger.csv", index=False)
json.dump(comparison.round(4).to_dict(orient="records"), open("artifacts/agent_comparison.json", "w"), indent=2)

lines = ["# RetainAI Agent — Example Transcripts\n", f"Model: `{AGENT_MODEL}`\n"]
for s, res in zip(scenarios, agent_runs):
    out = res["output"] or {}
    lines += [f"## {s['name']}\n", f"**Customer message:** {s['message'] or '_(proactive outreach)_'}\n",
              "**Tool calls:** " + " → ".join(t["tool"] for t in res["trace"]) + "\n",
              f"**Offer:** `{out.get('offer_code')}` | **Escalate:** {out.get('escalate_to_human')}\n",
              f"**Internal reasons:** {', '.join(out.get('key_reasons', []))}\n",
              f"> {out.get('customer_message')}\n"]
open("docs/agent_examples.md", "w").write("\n".join(lines))
print(comparison.round(3).to_string(index=False))

           model  compliance_rate  avg_judge_score  avg_latency_s  avg_input_tokens  avg_output_tokens  avg_tool_calls
nova-2-lite-v1:0              0.9            3.625          1.923            4343.6              223.0             3.8
 nova-micro-v1:0              1.0            3.425          3.594            5124.2              682.0             4.4
